[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghodsizadeh/tehran-stocks/blob/master/Example/Technical%20Analysis.ipynb)
[![Documentation](https://img.shields.io/badge/Docs-%D9%85%D8%B3%D8%AA%D9%86%D8%AF%D8%A7%D8%AA%20%D9%81%D8%A7%D8%B1%D8%B3%DB%8C-0d9488.svg)](https://ghodsizadeh.github.io/tehran-stocks/)
[![GitHub stars](https://img.shields.io/github/stars/ghodsizadeh/tehran-stocks?style=social)](https://github.com/ghodsizadeh/tehran-stocks)

# Tehran Stocks v3 — Adjusted Technical Analysis & Jalali Candles

Compute **backward-adjusted** technical indicators (SMA, EMA, Bollinger Bands, RSI-14, MACD), overlay daily retail money flow (`ورود پول حقیقی`), and aggregate calendar-accurate **Jalali monthly/yearly OHLCV bars** without broken legacy dependencies.

> **Note:** All TSETMC prices and trade values are in **Iranian Rials (IRR)** (`1 Toman = 10 IRR`; `1 Billion Toman = 1e10 IRR`).

## 1. Installation & Imports

In [ ]:
%pip install -q "tehran-stocks>=3,<4" matplotlib

In [ ]:
import jalali_pandas  # registers .jalali Series/DataFrame accessors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tehran_stocks as ts

plt.rcParams["figure.figsize"] = (14, 10)
plt.rcParams["axes.grid"] = True

## 2. Raw vs. Backward-Adjusted Price History

In TSETMC, dividends (`مجمع`) and capital increases (`افزایش سرمایه`) create sharp unadjusted price drops. Passing `adjusted=True` automatically applies `GetPriceAdjustList` factors backward.

In [ ]:
SYMBOL = "کگل"

with ts.Client() as client:
    raw_bars = client.prices.history(SYMBOL, days=400, adjusted=False)
    adj_bars = client.prices.history(SYMBOL, days=400, adjusted=True)
    adjustments = client.prices.adjustments(SYMBOL)
    ct_history = client.client_type.history(SYMBOL, days=400)

df_raw = ts.to_dataframe(raw_bars, jalali=True, index_by_date=True)
df = ts.to_dataframe(adj_bars, jalali=True, index_by_date=True)

print(f"Loaded {len(df)} adjusted trading days and {len(adjustments)} corporate-action adjustments.")
for adj in adjustments[:5]:
    print(
        f"  {adj.date}: raw={adj.unadjusted_close:,.0f} -> "
        f"adj={adj.adjusted_close:,.0f} (ratio={adj.ratio:.4f})"
    )

## 3. Calendar-Accurate Jalali Monthly OHLCV Bars

Avoid Gregorian `resample('ME')` on Iranian stocks (month boundaries shift by ~9–11 days). Group directly on `JalaliDatetimeIndex` components (`year`, `month`) with explicit OHLCV aggregations:

In [ ]:
j_year = df.index.year.astype(int)
j_month = df.index.month.astype(int)
df["jdate_str"] = df["date"].jalali.to_jalali().jalali.strftime("%Y-%m-%d")

monthly = df.groupby([j_year, j_month]).agg(
    open=("open", "first"),
    high=("high", "max"),
    low=("low", "min"),
    close=("close", "last"),
    volume=("volume", "sum"),
    value=("value", "sum"),
)
monthly.index.names = ["j_year", "j_month"]
monthly["return_pct"] = (monthly["close"].pct_change() * 100).round(2)
monthly["value_bt"] = (monthly["value"] / 1e10).round(1)

monthly.tail(12)

## 4. Technical Indicators (SMA, Bollinger Bands, RSI-14, MACD) + Real Money Flow

In [ ]:
close = df["close"].astype(float)

# Moving averages & Bollinger Bands (20, 2)
df["sma_20"] = close.rolling(20).mean()
df["sma_50"] = close.rolling(50).mean()
std_20 = close.rolling(20).std()
df["bb_upper"] = df["sma_20"] + 2 * std_20
df["bb_lower"] = df["sma_20"] - 2 * std_20

# Wilder-style RSI (14)
delta = close.diff()
gain = delta.clip(lower=0.0)
loss = (-delta).clip(lower=0.0)
avg_gain = gain.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
rs = avg_gain / avg_loss.replace(0.0, np.nan)
df["rsi_14"] = 100.0 - (100.0 / (1.0 + rs))

# MACD (12, 26, 9)
ema_12 = close.ewm(span=12, adjust=False).mean()
ema_26 = close.ewm(span=26, adjust=False).mean()
df["macd"] = ema_12 - ema_26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd"] - df["macd_signal"]

# Join daily real money flow (Billion Toman) & buyer power
ct_rows = [
    {
        "date": rec.date,
        "real_flow_bt": rec.real_money_flow / 1e10,
        "buyer_power": rec.buyer_power,
    }
    for rec in ct_history
]
df_ct = pd.DataFrame(ct_rows)
if not df_ct.empty:
    df_ct["date"] = pd.to_datetime(df_ct["date"])
    df = df.merge(df_ct, on="date", how="left").set_axis(df.index)

df[["jdate_str", "close", "sma_20", "rsi_14", "macd", "real_flow_bt", "buyer_power"]].tail(10)

## 5. Multi-Panel Technical Dashboard (Price + Real Money Flow + RSI + MACD)

In [ ]:
plot_df = df.tail(180).copy()
x = np.arange(len(plot_df))
tick_step = max(len(plot_df) // 10, 1)
tick_pos = x[::tick_step]
tick_labels = plot_df["jdate_str"].iloc[::tick_step]

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True, gridspec_kw={"height_ratios": [3, 1.3, 1.2, 1.3]})

# Panel 1: Adjusted Close + SMAs + Bollinger Bands
axes[0].plot(x, plot_df["close"], label="Adjusted Close (IRR)", color="#0f766e", linewidth=1.8)
axes[0].plot(x, plot_df["sma_20"], label="SMA 20", color="#f59e0b", linewidth=1.1)
axes[0].plot(x, plot_df["sma_50"], label="SMA 50", color="#6366f1", linewidth=1.1)
axes[0].fill_between(x, plot_df["bb_lower"], plot_df["bb_upper"], color="#99f6e4", alpha=0.25, label="Bollinger (20, 2)")
axes[0].set_title(f"{SYMBOL} — Adjusted Price, Real Money Flow, RSI & MACD (Jalali Dates)")
axes[0].set_ylabel("Price (IRR)")
axes[0].legend(loc="upper left")

# Panel 2: Real Money Flow (Billion Toman)
flow_vals = plot_df.get("real_flow_bt", pd.Series(0.0, index=plot_df.index)).fillna(0.0)
flow_colors = np.where(flow_vals >= 0, "#16a34a", "#dc2626")
axes[1].bar(x, flow_vals, color=flow_colors, alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Real Flow (B Toman)")

# Panel 3: RSI (14)
axes[2].plot(x, plot_df["rsi_14"], color="#2563eb", linewidth=1.3)
axes[2].axhline(70, linestyle="--", color="#dc2626", linewidth=0.9)
axes[2].axhline(30, linestyle="--", color="#16a34a", linewidth=0.9)
axes[2].set_ylim(0, 100)
axes[2].set_ylabel("RSI (14)")

# Panel 4: MACD
hist_vals = plot_df["macd_hist"].fillna(0.0)
axes[3].bar(x, hist_vals, color=np.where(hist_vals >= 0, "#16a34a", "#dc2626"), alpha=0.5)
axes[3].plot(x, plot_df["macd"], label="MACD", color="#1d4ed8", linewidth=1.2)
axes[3].plot(x, plot_df["macd_signal"], label="Signal", color="#ea580c", linewidth=1.1)
axes[3].set_ylabel("MACD")
axes[3].legend(loc="upper left")
axes[3].set_xticks(tick_pos)
axes[3].set_xticklabels(tick_labels, rotation=30, ha="right")

plt.tight_layout()
plt.show()